In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sensofit.package_loader import load_experiment, load_package

In [5]:
cxw_file = "/mnt/c/Users/kgr26424/Documents/Creoptix_test/export/20260318_EV71 2A Binding assay.cxw"
data_cxw = load_experiment(cxw_file)

In [6]:
zip_file = "/mnt/c/Users/kgr26424/Documents/Creoptix_test/export/20260318_EV71_2A_Binding_assay.zip"
data_zip = load_experiment(zip_file)

No experiment name given; loading first in alphabetical order: '20260318_EV71_2A_Binding_assay'
Other available experiments in package: []


In [7]:
print(data_cxw.keys() == data_zip.keys())

True


In [8]:
samples_cxw      = data_cxw['samples']
blanks_cxw       = data_cxw['blanks']
dmso_cals_cxw    = data_cxw['dmso_cals']
other_cycles_cxw = data_cxw.get('other_cycles', [])

In [9]:
samples_zip      = data_zip['samples']
blanks_zip       = data_zip['blanks']
dmso_cals_zip    = data_zip['dmso_cals']
other_cycles_zip = data_zip.get('other_cycles', [])

In [10]:
print(len(samples_cxw) == len(samples_zip), len(samples_cxw), len(samples_zip))
print(len(blanks_cxw) == len(blanks_zip), len(blanks_cxw), len(blanks_zip))
print(len(dmso_cals_cxw) == len(dmso_cals_zip), len(dmso_cals_cxw), len(dmso_cals_zip))
print(len(other_cycles_cxw) == len(other_cycles_zip), len(other_cycles_cxw), len(other_cycles_zip))

True 849 849
True 348 348
True 84 84
True 336 336


In [11]:
print(samples_cxw[0].keys() == samples_zip[0].keys())
print(samples_cxw[0].keys())

True
dict_keys(['index', 'cycle_type', 'guid', 'name', 'slot', 'slot_side', 'compound', 'concentration_M', 'mw', 'reagent_category', 'reagent_volume_uL', 'markers', 'flow_rate_uLmin', 'contact_time_s', 'time_after_injection_s', 'baseline_duration_s', 'injection_mode', 'pulse_durations_s', 'chip_prime_mode', 'wash_mode', 'buffer_inlet', 'block_id', 'state', 'enabled', 'time', 'signal', 'raw_active', 'raw_reference', 'channel'])


In [12]:
similar_counts = {"active": 0, "reference": 0, "signal": 0}
for s_cxw, s_zip in zip(samples_cxw, samples_zip):
    for key in s_cxw.keys():
        if key not in ("raw_active", "raw_reference", "signal"):
            if type(s_cxw[key]) != np.ndarray:
                assert s_cxw[key] == s_zip[key], f"Mismatch in key {key}: {s_cxw[key]} vs {s_zip[key]}"
            else:
                assert np.allclose(s_cxw[key], s_zip[key]), f"Mismatch in key {key}: {s_cxw[key]} vs {s_zip[key]}"
    check_active = s_cxw['raw_active'] - s_zip['raw_active']
    if not np.allclose(check_active, 0):
        print(f"Average difference in raw_active: {np.mean(check_active)}")
    else:
        similar_counts["active"] += 1
    check_reference = s_cxw['raw_reference'] - s_zip['raw_reference']
    if not np.allclose(check_reference, 0):
        print(f"Average difference in raw_reference: {np.mean(check_reference)}")
    else:
        similar_counts["reference"] += 1
    check_signal = s_cxw['signal'] - s_zip['signal']
    if not np.allclose(check_signal, 0):
        print(f"Average difference in signal: {np.mean(check_signal)}")
    else:
        similar_counts["signal"] += 1

print("Similar counts:")
for category, count in similar_counts.items():
    print(f"  {category.capitalize()}: {count}")

Average difference in raw_active: -4.7999744613965356e-09
Average difference in raw_reference: -6.341560704944034e-09
Average difference in signal: -4.325038753449917e-09
Average difference in raw_reference: -6.341560704944034e-09
Average difference in signal: 5.275414635737737e-09
Average difference in raw_reference: -6.341560704944034e-09
Average difference in signal: 5.2753215034802756e-09
Average difference in raw_active: 2.1334970369935035e-09
Average difference in raw_reference: 7.191821108184134e-09
Average difference in signal: -5.591614171862602e-09
Average difference in raw_reference: 7.191821108184134e-09
Average difference in signal: -6.657217939694722e-09
Average difference in raw_reference: 7.191821108184134e-09
Average difference in signal: -6.657124807437261e-09
Average difference in raw_active: -2.6653675983349483e-10
Average difference in raw_reference: 1.4334776399967572e-09
Average difference in signal: 5.766803709169229e-09
Average difference in raw_reference: 1.43

In [13]:
print("Looking at signal value differences for the first sample:")
for v_cxw, v_zip in zip(samples_cxw[0]["signal"], samples_zip[0]["signal"]):
    print(f"cxw: {v_cxw}, zip: {v_zip}, diff: {v_cxw - v_zip}")

Looking at signal value differences for the first sample:
cxw: -73564.42553710938, zip: -73564.425537, diff: -1.0937219485640526e-07
cxw: -73564.43041992188, zip: -73564.43042, diff: 7.812923286110163e-08
cxw: -73564.43872070312, zip: -73564.438721, diff: 2.9687362257391214e-07
cxw: -73564.3115234375, zip: -73564.311523, diff: -4.375033313408494e-07
cxw: -73564.142578125, zip: -73564.142578, diff: -1.2500095181167126e-07
cxw: -73563.99047851562, zip: -73563.990479, diff: 4.84375050291419e-07
cxw: -73563.81762695312, zip: -73563.817627, diff: 4.687171895056963e-08
cxw: -73563.77368164062, zip: -73563.773682, diff: 3.5937409847974777e-07
cxw: -73563.77661132812, zip: -73563.776611, diff: -3.2813113648444414e-07
cxw: -73563.77490234375, zip: -73563.774902, diff: -3.4374534152448177e-07
cxw: -73563.77124023438, zip: -73563.77124, diff: -2.3437314666807652e-07
cxw: -73563.79614257812, zip: -73563.796143, diff: 4.218745743855834e-07
cxw: -73563.83081054688, zip: -73563.830811, diff: 4.531320